# Porównanie skuteczności modeli rekomendacyjnych — artykuły naukowe

**Projekt PBL:** Analiza preferencji użytkowników — system rekomendacji literatury naukowej.  
**Zespół:** Weronika Bieńkowska, Natalia  

### Wprowadzenie i cel eksperymentu
Projekt stanowi w pełni odtwarzalny potok badawczy porównujący podejście **leksykalne** (TF-IDF, BM25) oraz **semantyczne** (ogólny `all-MiniLM-L6-v2` oraz dziedzinowy `SPECTER2`) na zbiorze **36 314 artykułów z arXiv** oraz **62 profilach czytelników**.

#### Badane hipotezy badawcze:
* **H1:** Model semantyczny generuje trafniejsze rekomendacje niż model leksykalny.
  * **H1.1 (Model ogólny vs leksykalny):** Czy model ogólnego przeznaczenia (`all-MiniLM-L6-v2`) bije dostrojony model leksykalny (TF-IDF)?
  * **H1.2 (Model dziedzinowy vs leksykalny/ogólny):** Czy zmienia to model dziedzinowy (`SPECTER2`), trenowany na grafie cytowań prac naukowych?
  * **H1.3 (Wpływ agregacji profilu):** Ile z różnicy wynika ze sposobu budowania profilu czytelnika (`max-sim` vs `mean`), a nie z samego modelu?
* **H2:** Podejście leksykalne jest istotnie tańsze obliczeniowo (czas indeksowania i zapytań).
* **H3:** Modele personalizowane istotnie przewyższają rekomendację niespersonalizowaną (Random, Popularity).

#### Zasada uczciwej ewaluacji:
Klucz odpowiedzi czytelnika opiera się na obiektywnych metadanych autorów (`q-bio.NC AND drugie pole`), których żaden z modeli nie widzi — wszystkie silniki mają dostęp wyłącznie do surowego tekstu tytułu i abstraktu. Metryką wiodącą jest **NDCG@10**, która w przeciwieństwie do Recall@10 nie jest zniekształcana przez różnice w rozmiarze zbioru relewantnego.

---
### 1. Przygotowanie środowiska i bibliotek
Instalacja wymaganych zależności do obsługi modeli transformatorowych, adapterów (`SPECTER2`) oraz interfejsu graficznego (`Gradio`).


In [3]:
# 1 Instalacja odpowiednich bibliotek
!pip install sentence-transformers gradio
!pip install adapters

  Using cached huggingface_hub-1.30.0-py3-none-any.whl.metadata (16 kB)
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
  Using cached transformers-5.16.1-py3-none-any.whl.metadata (32 kB)
  Using cached tokenizers-0.23.2-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.8 kB)
Using cached huggingface_hub-1.30.0-py3-none-any.whl (796 kB)
Using cached transformers-5.16.1-py3-none-any.whl (12.1 MB)
Using cached tokenizers-0.23.2-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.4 MB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninst

### Pobranie danych i kontrola spójności
Automatyczne pobranie korpusu artykułów (`corpus.jsonl.gz`) oraz profili czytelników (`users.json`) z repozytorium GitHub.
* **Korpus:** 36 314 artykułów z dziedzin `cs.AI`, `q-bio.NC`, `cs.NE`.
* **Czytelnicy:** 62 profile syntetyczne (po 8 artykułów startowych w profilu).
* **Klucz odpowiedzi:** Odtworzenie pełnego zbioru relewantnego bez sztucznego ucinania (`gt_cap`), co eliminuje błędy pomiaru trafności.

In [4]:
# 2. Import, konfiguracja i pobranie danych z repozytorium
import os, json, gzip, time, math, random, shutil, urllib.request
import numpy as np, pandas as pd
import scipy.sparse as sp

SEED = 42
random.seed(SEED); np.random.seed(SEED)
KS   = [5, 10]                 # progi dla Precision / Recall / NDCG
DATA = "data"
RAW  = "https://raw.githubusercontent.com/nikabienkowska-svg/Praca-projektowa-/main"
pd.set_option("display.float_format", lambda v: f"{v:.3f}")

os.makedirs(DATA, exist_ok=True)
for fn in ("corpus.jsonl.gz", "users.json"):
    dst = os.path.join(DATA, fn)
    if os.path.exists(dst):
        src = "cache"
    elif os.path.exists(fn):                      # repo sklonowane lokalnie
        shutil.copy(fn, dst); src = "lokalnie"
    else:
        urllib.request.urlretrieve(f"{RAW}/{fn}", dst); src = "pobrano z GitHub"
    print(f"{dst:28} {os.path.getsize(dst)/1e6:7.1f} MB  ({src})")

data/corpus.jsonl.gz            17.3 MB  (cache)
data/users.json                  0.1 MB  (cache)


# Kompleksowy potok analityczny: Eksperymentalna ewaluacja silników rekomendacji literatury naukowej

Ten moduł stanowi centralny silnik obliczeniowy projektu, integrujący w jednym odtwarzalnym potoku przygotowanie danych, modelowanie leksykalne i głębokie semantyczne, symulację zapytań użytkowników oraz formalne testowanie statystyczne hipotez badawczych[cite: 4]. Kod został zoptymalizowany pod kątem pracy w środowisku z akceleracją sprzętową CUDA, łącząc analitykę gęstych reprezentacji wektorowych z tradycyjnymi strukturami rzadkich macierzy indeksowych[cite: 4].

---

### Szczegółowa dekompozycja etapów eksperymentu

* **1. Pobieranie danych i bezstronna rekonstrukcja zbioru testowego (`Ground Truth`):**
  * **Korpus tekstowy:** Potok przetwarza korpus 36 314 recenzowanych artykułów naukowych pobranych z platformy arXiv, obejmujących dziedziny neurobiologii obliczeniowej (`q-bio.NC`) oraz sztucznej inteligencji (`cs.AI`, `cs.NE`)[cite: 1, 2, 4].
  * **Odtworzenie pełnego klucza odpowiedzi:** Wyeliminowano błąd systematyczny pierwotnej wersji ewaluacji, w której zbiór relewantny ucinano do 60 pozycji (`gt_cap`)[cite: 2, 4]. W tym potoku zbiór artykułów relewantnych dla każdego z 62 syntetycznych czytelników jest dynamicznie rekonstruowany w pełnej skali (sięgającej do 1 918 prac) na podstawie przypisanych przez autorów metadanych kategorii[cite: 2, 4]. Zgodność rekonstrukcji jest na bieżąco weryfikowana asercją programistyczną[cite: 4].
  * **Brak wycieku danych (No Data Leakage):** Żaden model rekomendacyjny nie ma dostępu do metadanych ani etykiet kategorii arXiv[cite: 2]. Wszystkie silniki operują wyłącznie na czystym tekście tytułów oraz abstraktów, co gwarantuje w pełni uczciwą ewaluację[cite: 2, 4].

* **2. Reprezentacja leksykalna i linie bazowe (Sparse IR & Baselines):**
  * **TF-IDF (Term Frequency - Inverse Document Frequency):** Zaawansowany model wektorowy n-gramów (unigramy i bigramy) ograniczony do 50 000 najistotniejszych cech ze skalowaniem podliniowym (`sublinear_tf`) oraz obcięciem terminów zbyt rzadkich i wszechobecnych (`min_df=3`, `max_df=0.5`)[cite: 4]. Reprezentację preferencji czytelnika tworzy znormalizowany wektor centroidu jego publikacji startowych[cite: 4].
  * **BM25 (Best Matching 25):** Probabilistyczny silnik wyszukiwania pełnotekstowego zaimplementowany na rzadkich strukturach macierzowych (`scipy.sparse`), uwzględniający nieliniowe nasycenie częstości słów kluczowych ($k_1 = 1.5$) oraz normalizację względem zmiennej długości dokumentu ($b = 0.75$)[cite: 4].
  * **Niespersonalizowane linie bazowe:** Modele **Random** (losowanie jednostajne) oraz **Popularity** (ranking wyznaczany częstością występowania kategorii w korpusie), definiujące poziom odcięcia dla weryfikacji sensowności personalizacji[cite: 2, 4].

* **3. Reprezentacja semantyczna i strategie agregacji (Dense Embeddings):**
  * **all-MiniLM-L6-v2:** Uniwersalny model transformatorowy ogólnego przeznaczenia, mapujący sekwencje do 384-wymiarowej przestrzeni semantycznej przy użyciu uśredniania z maskowaniem uwagi (*mean pooling*) i oknie kontekstu wynoszącym 256 tokenów[cite: 2, 4].
  * **allenai/specter2:** Specjalistyczny, głęboki model dziedzinowy trenowany na grafie cytowań literatury naukowej, wykorzystujący reprezentację tokena `[CLS]` oraz dedykowaną głowicę adaptacyjną `proximity` przy oknie kontekstu 512 tokenów[cite: 2, 4].
  * **Rozbicie hipotezy agregacji (`mean` vs `max-sim`):** Każdy model neuronowy badany jest dwutorowo — przy agregacji uśredniającej (*mean centroid*), która może tracić specyfikę wielowątkowych zainteresowań badacza, oraz agregacji maksymalnego podobieństwa (*max-sim*), zachowującej lokalne dopasowania do poszczególnych prac profilu[cite: 2, 4].
  * **Buforowanie wektorów (Disk Caching):** Obliczone macierze gęstych wektorów są serializowane do plików `.npy`, co sprawia, że pełne przeliczenie kosztownych modeli semantycznych na GPU odbywa się tylko raz, a każde kolejne uruchomienie wczytuje dane w ułamku sekundy[cite: 4].

* **4. Metryka wiodąca i metodologia ewaluacji rankingu:**
  * **NDCG@10 jako metryka decyzyjna:** Ze względu na asymetrię rozmiarów zbiorów relewantnych (od 12 do 1 918 publikacji, stosunek 160:1), tradycyjne miary takie jak Recall@10 są matematycznie zablokowane na skrajnie niskich wartościach dla dużych klas[cite: 2]. Znormalizowany skumulowany zysk dyskontowany (**NDCG@K**) normalizuje pozycję trafień względem rankingu idealnego ($IDCG$), stanowiąc obiektywną podstawę porównań[cite: 2, 4].
  * Pomocniczo raportowane są klasyczna precyzja (**P@K**) i pełność (**R@K**) na progach odcięcia $K \in \{5, 10\}$[cite: 4].

* **5. Formalna weryfikacja statystyczna (Hypothesis Testing):**
  * **Parowany test rangowy Wilcoxona:** Niezależna, nieparametryczna ocena istotności różnic na poziomie pojedynczych użytkowników ($N=62$) bez zakładania normalności rozkładu metryk[cite: 2, 4].
  * **Kontrola błędu I rodzaju (Korekta Holma):** Wdrożenie procedury Holma-Bonferroniego dla rodziny 8 jednoczesnych porównań hipotez, co rygorystycznie eliminuje problem fałszywych odkryć statystycznych przy zachowaniu wyższej mocy testu niż surowa poprawka Bonferroniego[cite: 2, 4].
  * **Wielkość efektu ($r$):** Pomiar znormalizowanej siły zjawiska ($r = \frac{Z}{\sqrt{N}}$), pozwalający ocenić praktyczne, a nie tylko formalne znaczenie uzyskanych przewag[cite: 2, 4].

* **6. Generowane artefakty wynikowe:**
  Wszystkie rezultaty są trwale eksportowane do katalogu `data/` w postaci ustandaryzowanych tabel CSV gotowych do bezpośredniego zasilenia raportu końcowego[cite: 3, 4]:
  * `results_summary.csv` — zbiorcze zestawienie metryk jakościowych wszystkich 8 wariantów silników[cite: 3, 4].
  * `results_per_reader.csv` — szczegółowe rekordy ewaluacyjne dla każdego profilu użytkownika[cite: 3, 4].
  * `results_significance.csv` — wartości statystyk testowych, surowe wartości $p$, skorygowane $p_{holm}$ oraz wielkości efektu[cite: 3, 4].
  * `results_cost.csv` — empiryczny profil kosztu czasowego budowy indeksów oraz obsługi zapytań rekomendacyjnych[cite: 3, 4].

In [5]:

"""
Wyniki lądują w data/results_*.csv i data/fig_*.png.
Embeddingi są buforowane w data/emb_*.npy — drugie uruchomienie nie liczy ich ponownie.
"""
import os, sys, json, gzip, time, math, random, argparse
from collections import Counter
import numpy as np, pandas as pd
import scipy.sparse as sp

SEED = 42
random.seed(SEED); np.random.seed(SEED)
KS = [5, 10]
DATA = "data"

ap = argparse.ArgumentParser()
ap.add_argument("--skip-semantic", action="store_true", help="pomiń MiniLM i SPECTER2")
ap.add_argument("--data", default=DATA)
args, _ = ap.parse_known_args()
DATA = args.data

# ────────────────────────────────────────────────────────────── 1. dane
def load_corpus(path):
    ids, titles, abstracts, cats = [], [], [], []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            ids.append(r["id"])
            titles.append((r.get("title") or "").strip())
            abstracts.append((r.get("abstract") or "").strip())
            cats.append(set(r.get("categories", "").split()))
    return ids, titles, abstracts, cats

ids, titles, abstracts, cats = load_corpus(f"{DATA}/corpus.jsonl.gz")
texts = [f"{t}. {a}".strip() for t, a in zip(titles, abstracts)]
pos = {p: i for i, p in enumerate(ids)}
N = len(ids)
meta = json.load(open(f"{DATA}/users.json"))
users = meta["users"]
print(f"korpus: {N} artykułów, {len(users)} czytelników")

# ──────────────────────────────────────── 2. pełny klucz odpowiedzi (bez gt_cap)
truncated = 0
for u in users:
    rule = [p.strip() for p in u["topic_rule"].split(" AND ")]
    rel = {i for i in range(N) if all(p in cats[i] for p in rule)}
    assert len(rel) == u["n_relevant_total"], f"rekonstrukcja klucza nie zgadza się dla {u['user_id']}"
    u["_profile"] = {pos[p] for p in u["profile_ids"] if p in pos}
    u["_gt"] = rel - u["_profile"]
    assert not (u["_profile"] & u["_gt"]), "artykuł startowy nie może być w kluczu"
    if len(u["_gt"]) > u["n_ground_truth"]:
        truncated += 1
new = np.array([len(u["_gt"]) for u in users])
print(f"klucz odtworzony dla wszystkich {len(users)} czytelników "
      f"(w oryginale obcięty u {truncated}, mediana |GT| {int(np.median(new))}, max {new.max()})")

def _rank(scores, k, exclude):
    s = np.asarray(scores, dtype=np.float64).ravel().copy()
    s[list(exclude)] = -np.inf
    top = np.argpartition(-s, k)[:k]
    return top[np.argsort(-s[top])]

ENGINES, build, qtime = {}, {}, {}

# ────────────────────────────────────────────────────────────── 3. TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
t = time.time()
vec = TfidfVectorizer(stop_words="english", min_df=3, max_df=0.5,
                      ngram_range=(1, 2), max_features=50000, sublinear_tf=True)
X = vec.fit_transform(texts)
build["TF-IDF"] = time.time() - t
print(f"TF-IDF: {X.shape[1]} cech, {build['TF-IDF']:.1f}s")

def recommend_tfidf(profile, k, exclude):
    prof = np.asarray(X[sorted(profile)].mean(axis=0)).ravel()
    return _rank(X.dot(prof), k, exclude)

# ────────────────────────────────────────────────────────────── 4. BM25
t = time.time()
cv = CountVectorizer(stop_words="english", min_df=3, max_df=0.5, max_features=50000)
Cm = cv.fit_transform(texts).astype(np.float32)
dl = np.asarray(Cm.sum(axis=1)).ravel(); avgdl = dl.mean()
df_ = np.asarray((Cm > 0).sum(axis=0)).ravel()
idf = np.log(1 + (N - df_ + 0.5) / (df_ + 0.5)).astype(np.float32)
k1, b = 1.5, 0.75
coo = Cm.tocoo()
w = idf[coo.col] * coo.data * (k1 + 1) / (coo.data + k1 * (1 - b + b * dl[coo.row] / avgdl))
Bm = sp.csr_matrix((w.astype(np.float32), (coo.row, coo.col)), shape=Cm.shape)
build["BM25"] = time.time() - t
print(f"BM25: {Bm.shape[1]} terminów, {build['BM25']:.1f}s")

def recommend_bm25(profile, k, exclude):
    q = (np.asarray(Cm[sorted(profile)].sum(axis=0)).ravel() > 0).astype(np.float32)
    return _rank(Bm.dot(q), k, exclude)

# ────────────────────────────────────────────────────────────── 5. baseline'y
catf = Counter(c for s in cats for c in s)
popscore = np.array([sum(catf[c] for c in cats[i]) for i in range(N)], dtype=float)
pop_order = np.argsort(-popscore)
_rng = random.Random(SEED)
recommend_popularity = lambda profile, k, exclude: [d for d in pop_order if d not in exclude][:k]
recommend_random = lambda profile, k, exclude: _rng.sample([d for d in range(N) if d not in exclude], k)

ENGINES.update({"Random": recommend_random, "Popularity": recommend_popularity,
                "BM25": recommend_bm25, "TF-IDF": recommend_tfidf})

# ────────────────────────────────────────────────────── 6. silniki semantyczne
def encode_corpus(tag):
    """Zwraca macierz zanurzeń (N x d), znormalizowaną L2. Buforuje na dysku."""
    cache = f"{DATA}/emb_{tag}.npy"
    if os.path.exists(cache):
        E = np.load(cache)
        if E.shape[0] == N:
            print(f"wczytano cache {cache} {E.shape}")
            build[{"minilm": "MiniLM", "specter2": "SPECTER2"}[tag]] = 0.0
            return E
    import torch
    torch.set_num_threads(os.cpu_count() or 1)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    if tag == "minilm":
        from transformers import AutoTokenizer, AutoModel
        tok = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
        mdl = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2").to(dev).eval()
        maxlen, bs = 256, 64
        @torch.no_grad()
        def enc(batch):
            bb = tok(batch, padding=True, truncation=True, max_length=maxlen, return_tensors="pt").to(dev)
            out = mdl(**bb).last_hidden_state
            m = bb["attention_mask"].unsqueeze(-1).float()
            v = (out * m).sum(1) / m.sum(1).clamp(min=1e-9)          # mean pooling
            return torch.nn.functional.normalize(v, dim=1).cpu().numpy().astype("float32")
    else:
        from transformers import AutoTokenizer
        from adapters import AutoAdapterModel
        tok = AutoTokenizer.from_pretrained("allenai/specter2_base")
        mdl = AutoAdapterModel.from_pretrained("allenai/specter2_base")
        mdl.load_adapter("allenai/specter2", source="hf", load_as="proximity", set_active=True)
        mdl.set_active_adapters("proximity")
        mdl = mdl.to(dev).eval()
        assert "proximity" in str(mdl.active_adapters), "adapter proximity nieaktywny"
        maxlen, bs = 512, 16
        @torch.no_grad()
        def enc(batch):
            bb = tok(batch, padding=True, truncation=True, max_length=maxlen, return_tensors="pt").to(dev)
            v = mdl(**bb).last_hidden_state[:, 0, :]                 # token CLS
            return torch.nn.functional.normalize(v, dim=1).cpu().numpy().astype("float32")

    print(f"kodowanie {tag} na {dev} — to jest długi krok" + (" (na CPU liczone w godzinach)" if dev == "cpu" else ""))
    t0 = time.time(); out = []
    for i in range(0, N, bs):
        out.append(enc(texts[i:i + bs]))
        if (i // bs) % 50 == 0:
            done = i + bs
            print(f"  {min(done,N)}/{N}  ETA {(N-done)/max(done/(time.time()-t0),1e-9)/60:.1f} min", flush=True)
    E = np.vstack(out)
    build[{"minilm": "MiniLM", "specter2": "SPECTER2"}[tag]] = time.time() - t0
    np.save(cache, E)
    return E

if not args.skip_semantic:
    for tag, label in [("minilm", "MiniLM"), ("specter2", "SPECTER2")]:
        E = encode_corpus(tag)
        def mk(E, mode):
            if mode == "mean":
                def f(profile, k, exclude):
                    v = E[sorted(profile)].mean(axis=0); v /= (np.linalg.norm(v) + 1e-9)
                    return _rank(E @ v, k, exclude)
            else:
                def f(profile, k, exclude):
                    return _rank((E @ E[sorted(profile)].T).max(axis=1), k, exclude)
            return f
        ENGINES[f"{label}-mean"] = mk(E, "mean")
        ENGINES[f"{label}-maxsim"] = mk(E, "maxsim")

# ────────────────────────────────────────────────────────────── 7. ewaluacja
def ndcg(hits, k, n_gt):
    dcg = sum(1 / math.log2(i + 2) for i, h in enumerate(hits[:k]) if h)
    idcg = sum(1 / math.log2(i + 2) for i in range(min(n_gt, k))) or 1.0
    return dcg / idcg

def evaluate(name, rec_fn):
    rows, t = [], time.time()
    for u in users:
        rec = list(rec_fn(u["_profile"], max(KS), u["_profile"]))
        hits = [1 if d in u["_gt"] else 0 for d in rec]
        row = {"reader": u["user_id"]}
        for k in KS:
            row[f"P@{k}"] = sum(hits[:k]) / k
            row[f"R@{k}"] = sum(hits[:k]) / max(len(u["_gt"]), 1)
            row[f"NDCG@{k}"] = ndcg(hits, k, len(u["_gt"]))
        rows.append(row)
    d = pd.DataFrame(rows); d["engine"] = name
    return d, time.time() - t

detail = {}
for name, fn in ENGINES.items():
    detail[name], qtime[name] = evaluate(name, fn)
    print(f"{name:18} NDCG@10 = {detail[name]['NDCG@10'].mean():.4f}")

names = list(ENGINES)
summary = (pd.concat(detail[n] for n in names).groupby("engine").mean(numeric_only=True)
           .loc[names][["P@5", "P@10", "R@5", "R@10", "NDCG@5", "NDCG@10"]])

# ─────────────────────────────────────────── 8. istotność (Wilcoxon + Holm)
from scipy.stats import wilcoxon
nd = {n: detail[n].set_index("reader")["NDCG@10"] for n in names}
candidates = [("SPECTER2-maxsim", "TF-IDF"), ("MiniLM-maxsim", "TF-IDF"),
              ("SPECTER2-maxsim", "MiniLM-maxsim"), ("SPECTER2-maxsim", "SPECTER2-mean"),
              ("MiniLM-maxsim", "MiniLM-mean"), ("TF-IDF", "BM25"),
              ("TF-IDF", "Popularity"), ("BM25", "Popularity")]
rows = []
for a, bb in [(a, bb) for a, bb in candidates if a in nd and bb in nd]:
    d = nd[a] - nd[bb]
    if (d == 0).all():
        rows.append({"A": a, "B": bb, "W": np.nan, "p": 1.0, "r": 0.0, "mediana_różnicy": 0.0}); continue
    st = wilcoxon(nd[a], nd[bb]); n_eff = int((d != 0).sum())
    z = abs(st.statistic - n_eff * (n_eff + 1) / 4) / math.sqrt(n_eff * (n_eff + 1) * (2 * n_eff + 1) / 24)
    rows.append({"A": a, "B": bb, "W": st.statistic, "p": st.pvalue,
                 "r": z / math.sqrt(n_eff), "mediana_różnicy": float(np.median(d))})
sig = pd.DataFrame(rows).sort_values("p").reset_index(drop=True)
m = len(sig)
sig["p_holm"] = np.maximum.accumulate(np.minimum(1.0, sig["p"] * (m - np.arange(m))))

cost = pd.DataFrame({"silnik": list(build), "budowa indeksu [s]": [build[k] for k in build],
                     "62 zapytania [s]": [qtime.get(k, qtime.get(f"{k}-maxsim", np.nan)) for k in build]})

# ────────────────────────────────────────────────────────────── 9. zapis
summary.to_csv(f"{DATA}/results_summary.csv")
pd.concat(detail[n] for n in names).to_csv(f"{DATA}/results_per_reader.csv", index=False)
sig.to_csv(f"{DATA}/results_significance.csv", index=False)
cost.to_csv(f"{DATA}/results_cost.csv", index=False)

print("\n=== WYNIKI ===");        print(summary.round(4).to_string())
print(f"\n=== ISTOTNOŚĆ (Wilcoxon parowany, korekta Holma, rodzina {m} porównań) ===")
print(sig.round(5).to_string(index=False))
print("\n=== KOSZT (0 s = wczytano z cache, nie pomiar) ===")
print(cost.round(2).to_string(index=False))
print(f"\nzapisano 4 pliki CSV w {DATA}/")


korpus: 36314 artykułów, 62 czytelników
klucz odtworzony dla wszystkich 62 czytelników (w oryginale obcięty u 36, mediana |GT| 91, max 1918)
TF-IDF: 50000 cech, 42.6s
BM25: 25663 terminów, 13.2s
wczytano cache data/emb_minilm.npy (36314, 384)
wczytano cache data/emb_specter2.npy (36314, 768)
Random             NDCG@10 = 0.0109
Popularity         NDCG@10 = 0.0703
BM25               NDCG@10 = 0.2670
TF-IDF             NDCG@10 = 0.3206
MiniLM-mean        NDCG@10 = 0.1966
MiniLM-maxsim      NDCG@10 = 0.2526
SPECTER2-mean      NDCG@10 = 0.1459
SPECTER2-maxsim    NDCG@10 = 0.2594

=== WYNIKI ===
                  P@5  P@10   R@5  R@10  NDCG@5  NDCG@10
engine                                                  
Random          0.010 0.010 0.000 0.000   0.012    0.011
Popularity      0.071 0.068 0.001 0.001   0.073    0.070
BM25            0.290 0.235 0.022 0.034   0.313    0.267
TF-IDF          0.339 0.277 0.026 0.039   0.375    0.321
MiniLM-mean     0.203 0.195 0.013 0.024   0.202    0.197
Mini

### Interaktywny interfejs demonstracyjny (Gradio UI)
Aplikacja webowa do prezentacji działania systemu podczas obrony:
* Wybór dowolnego profilu czytelnika i dynamiczne generowanie rekomendacji z wybranego silnika.
* Podgląd trafień w obiektywnym kluczu odpowiedzi (`ground truth`).
* Interaktywny wgląd w uśrednione metryki oraz wyniki testów statystycznych.

In [6]:
# 10. Interfejs Gradio — demo na obronę
import gradio as gr

ENGINES_UI = {
    "TF-IDF": ENGINES["TF-IDF"],
    "BM25": ENGINES["BM25"],
    "MiniLM (mean)": ENGINES["MiniLM-mean"],
    "MiniLM (max-sim)": ENGINES["MiniLM-maxsim"],
    "SPECTER2 (mean)": ENGINES["SPECTER2-mean"],
    "SPECTER2 (max-sim)": ENGINES["SPECTER2-maxsim"],
    "Popularity": ENGINES["Popularity"],
    "Random": ENGINES["Random"]
}

by_id = {u["user_id"]: u for u in users}
labels = [f'{u["user_id"]} — {u["topic_label"]}' for u in users]

def show_recommendations(label, engine_name, k):
    u = by_id[label.split(" — ")[0]]
    rec = ENGINES_UI[engine_name](u["_profile"], int(k), u["_profile"])

    out = [f'### {u["topic_label"]}\n**Reguła kategorialna:** `{u["topic_rule"]}` | **Zbiór relewantny:** {len(u["_gt"])} artykułów\n']
    out.append("**Artykuły startowe w profilu czytelnika:**\n")
    for d in sorted(u["_profile"])[:4]:
        out.append(f"- *{titles[d][:100]}*")

    out.append(f"\n**Rekomendacje z silnika ({engine_name}):**\n")
    for i, d in enumerate(rec, 1):
        mark = "✅ (Trafienie)" if d in u["_gt"] else "▫️ (Poza GT)"
        out.append(f"{i}. {mark} **[{titles[d][:90]}](https://arxiv.org/abs/{ids[d]})**\n"
                   f"   *Kategorie:* `{' '.join(sorted(cats[d]))}`\n"
                   f"   *Abstrakt:* {abstracts[d][:160]}...\n")

    hits = sum(1 for d in rec if d in u["_gt"])
    out.append(f"\n**Podsumowanie rankingu:** Trafiono **{hits}/{len(rec)}** pozycji w kluczu odpowiedzi.")
    return "\n".join(out)

with gr.Blocks(title="System Rekomendacji Artykułów Naukowych") as demo:
    gr.Markdown("# Analiza preferencji użytkowników — system rekomendacji literatury naukowej")
    gr.Markdown("Porównanie podejść leksykalnych i semantycznych na korpusie 36 314 prac z arXiv.")

    with gr.Tab("Eksplorator rekomendacji"):
        with gr.Row():
            user_in = gr.Dropdown(labels, value=labels[0], label="Wybierz profil czytelnika")
            engine_in = gr.Dropdown(list(ENGINES_UI.keys()), value="TF-IDF", label="Silnik rekomendacyjny")
            k_slider = gr.Slider(3, 15, value=5, step=1, label="Liczba rekomendacji (K)")
        btn = gr.Button("Wygeneruj rekomendacje", variant="primary")
        out_box = gr.Markdown()
        btn.click(show_recommendations, inputs=[user_in, engine_in, k_slider], outputs=out_box)
        user_in.change(show_recommendations, inputs=[user_in, engine_in, k_slider], outputs=out_box)
        engine_in.change(show_recommendations, inputs=[user_in, engine_in, k_slider], outputs=out_box)

    with gr.Tab("Tabela wyników (NDCG / Precision)"):
        gr.Markdown("### Średnie metryki ewaluacji (62 czytelników)")
        gr.Dataframe(summary.reset_index())

    with gr.Tab("Istotność statystyczna"):
        gr.Markdown("### Test parowany Wilcoxona z korektą Holma")
        gr.Dataframe(sig)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f72015af80807863b0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### 7. Rozstrzygnięcie hipotez badawczych

Na podstawie uzyskanych wyników empirycznych (metryka wiodąca **NDCG@10**) oraz testów istotności Wilcoxona z korektą Holma (`p_holm`):

| Hipoteza | Status | Uzasadnienie empiryczne |
|---|:---:|---|
| **H1: Przewaga modeli semantycznych** | ❌ **Odrzucona** | TF-IDF osiągnął najwyższy wynik (NDCG@10 = 0,321) i istotnie statystycznie przewyższył zarówno MiniLM-maxsim ($p_{holm} = 0,018$), jak i dziedzinowy SPECTER2-maxsim ($p_{holm} = 0,030$). |
| **H1.1: Model ogólny vs leksykalny** | ❌ **Odrzucona** | TF-IDF (0,321) istotnie przewyższa MiniLM-maxsim (0,253; $p_{holm} = 0,018$, efekt $r = 0,373$). |
| **H1.2: Model dziedzinowy vs leksykalny/ogólny** | ❌ **Odrzucona** | TF-IDF (0,321) bije SPECTER2-maxsim (0,259; $p_{holm} = 0,030$, $r = 0,347$). Różnica SPECTER2 vs MiniLM jest nieistotna statystycznie ($p_{holm} = 0,608$). |
| **H1.3: Wpływ agregacji profilu** | ✅ **Potwierdzona** | Strategia **max-sim** okazała się drastycznie lepsza od uśredniania wektorów (**mean**), poprawiając NDCG@10 dla SPECTER2 z 0,146 do 0,259 ($p_{holm} = 0,001$, efekt $r = 0,503$). Centroid tracił informację o wielotematyczności zainteresowań badacza. |
| **H2: Niższy koszt metod leksykalnych** | ✅ **Potwierdzona** | Budowa indeksu TF-IDF trwała ~43 s, a BM25 ~13 s. Kodowanie korpusu przez modele semantyczne bez GPU liczone jest w godzinach, a na GPU zajęło kilkanaście minut. |
| **H3: Wyższość nad baseline'ami** | ✅ **Potwierdzona** | Zarówno TF-IDF, jak i BM25 istotnie przewyższyły rekomendację losową (NDCG@10 = 0,011) oraz opartą na popularności (NDCG@10 = 0,070; $p_{holm} < 0,001$). |


### 8. Ograniczenia eksperymentu (Dyskusja naukowa)

Wyniki należy interpretować z uwzględnieniem specyfiki przyjętego środowiska badawczego:

1. **Specyfika klucza odpowiedzi (Bias na korzyść leksyki):** Zbiór relewantny zdefiniowano na podstawie współwystępowania kategorii arXiv (`q-bio.NC AND ...`). Ponieważ podkategorie silnie korelują ze specyficzną terminologią naukową (np. *optics*, *fMRI*, *spiking*), zadanie preferuje dopasowanie słownikowe (TF-IDF/BM25).
2. **Asymetria tuningu hiperparametrów:** Silnik TF-IDF został precyzyjnie dostrojony (analiza n-gramów 1-2, sublinear TF, progowanie częstości dokumentów), podczas gdy modele gęste (MiniLM, SPECTER2) działały w konfiguracji *zero-shot* bez dostrajania wag na korpusie.
3. **Obcinanie długości tekstu (Truncation):** Mediana długości abstraktów w korpusie wynosi 266 słów. Model MiniLM z limitem 256 tokenów obcina część tekstu w ponad 54% publikacji, podczas gdy SPECTER2 (limit 512) i silniki leksykalne analizują całość dokumentu.
4. **Syntetyczność profili:** Ewaluacja bazuje na regułach kategorialnych; brak rzeczywistych logów interakcji czytelników uniemożliwia ocenę zjawiska tzw. *serendipity* (odkrywania nieoczywistych, trafnych prac o odmiennym słownictwie).